In [5]:
def generate_portfolio(data, n=5):
    """
    Ranks settlements by AgroLink_Score and displays key investment metrics.
    """
    # Sort by impact
    portfolio = data.sort_values(by='AgroLink_Score', ascending=False).head(n)
    
    # Using the exact column names from your list
    cols = [
        'id', 
        'AgroLink_Score', 
        'Pop2025', 
        'Pop2030', 
        'PVHybridGenCapex2025', 
        'PVHybridGenLCOE2025'
    ]
    
    return portfolio[cols]

# Execution
sofia_top_5 = generate_portfolio(df)
print("--- SOFIA REGION: HIGH-IMPACT INVESTMENT PORTFOLIO ---")
display(sofia_top_5)

--- SOFIA REGION: HIGH-IMPACT INVESTMENT PORTFOLIO ---


,id,AgroLink_Score,Pop2025,Pop2030,PVHybridGenCapex2025,PVHybridGenLCOE2025
17202,125389,406909.098346,40340.605,49556.527,13143229.0,0.229371
17338,125732,305441.437446,22343.984,27448.530,8173025.5,0.242727
17436,126025,300059.299069,21780.203,26755.951,7096130.0,0.226706
16955,124976,282841.353088,31867.014,39147.120,10382478.0,0.232035
15551,36167,208327.075584,14767.551,18141.238,4811363.0,0.228038


In [7]:
def budget_search(data, limit):
    """
    Filters for sites under a specific Capex budget and ranks them by impact.
    """
    # Filter by the 2025 Capex column
    options = data[data['PVHybridGenCapex2025'] <= limit].copy()
    
    if options.empty:
        return "No settlements found within that budget."
    
    # Sort by score and pick top 3
    wins = options.sort_values(by='AgroLink_Score', ascending=False).head(3)
    
    # Using 'Pop2025' for immediate impact reporting
    return wins[['id', 'AgroLink_Score', 'PVHybridGenCapex2025', 'Pop2025']]

# Test it with a $2M budget
print(f"\n--- Top 3 sites for a $2,000,000 Budget ---")
display(budget_search(df, 2000000))


--- Top 3 sites for a $2,000,000 Budget ---


,id,AgroLink_Score,PVHybridGenCapex2025,Pop2025
17300,125659,65294.445348,1727860.1,5303.3335
16276,39837,59825.572891,1327781.6,3629.9817
17014,125066,59809.679397,1973074.2,6055.9710


### **Methodology: The AgroLink Proxy**
To identify optimal sites for productive use of energy (PUE), we utilize the **AgroLink Score**. This metric prioritizes settlements where high solar potential overlaps with significant community infrastructure and projected population growth.

**The Formula:**
$$\text{AgroLink Score} = \frac{(Pop_{2030} \times GHI \times (Anchors + 1))}{(\text{GridDist} + 1)}$$

**Where:**
* **$Pop_{2030}$**: Projected population, ensuring long-term infrastructure viability.
* **$GHI$**: Solar potential (Global Horizontal Irradiance).
* **$Anchors$**: Proximity to social infrastructure (Schools, Clinics, etc.).
* **$GridDist$**: Distance to medium-voltage lines, serving as a proxy for connection difficulty.

*Note: Scores are currently normalized to a scale of 0-100 based on the Sofia, Madagascar regional maximum.*


In [9]:
import openai
from getpass import getpass

# 1. Securely input your OpenAI API Key
api_key = getpass("Paste your OpenAI API Key (sk-...) here: ")
client = openai.OpenAI(api_key=api_key)

# 2. Test the connection to the GPT-4o brain
try:
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": "ChatOnSSET Engine Check: Are you ready to analyze Madagascar's energy data?"}],
        max_tokens=50
    )
    # This will print the AI's response if the connection is successful
    print(f"\n✅ AI Response: {response.choices[0].message.content}")
except Exception as e:
    # This will catch and explain the error if it still fails
    print(f"\n❌ Error: {e}")

Paste your OpenAI API Key (sk-...) here:  ········



✅ AI Response: Yes, I'm ready to help you analyze Madagascar's energy data. Please provide the specific data or questions you have, and I'll do my best to assist you.


In [10]:
import json

# 1. Define the System Personality
system_instruction = """
You are ChatOnSSET, a specialized Geospatial AI agent for energy planning in Madagascar.
Your goal is to help investors identify high-impact solar mini-grid sites using the 'AgroLink' framework.

You have access to two main tools:
- generate_portfolio: Use this for general requests for the 'best' or 'highest impact' sites.
- budget_search: Use this when the user mentions a specific dollar amount or budget.

When you present results, mention the id of the settlements and explain that they were chosen 
based on a balance of population growth (Pop2030), solar potential (GHI), and social infrastructure.
"""

# 2. Define the 'Manual' (Tool Specs)
tools = [
    {
        "type": "function",
        "function": {
            "name": "generate_portfolio",
            "description": "Finds the top N highest impact solar investment sites in Sofia, Madagascar.",
            "parameters": {
                "type": "object",
                "properties": {
                    "n": {"type": "integer", "description": "Number of sites to return. Default is 5."}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "budget_search",
            "description": "Finds the best electrification sites within a specific dollar budget.",
            "parameters": {
                "type": "object",
                "properties": {
                    "limit": {"type": "number", "description": "The budget limit in USD (e.g. 2000000)."}
                },
                "required": ["limit"]
            }
        }
    }
]

# 3. The Function Calling Loop
def chat_onsset(user_input):
    # Initial call to the AI
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_input}
        ],
        tools=tools,
        tool_choice="auto"
    )
    
    message = response.choices[0].message
    
    # Check if the AI wants to use your Python functions
    if message.tool_calls:
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            
            print(f"🛠️ AI DECISION: Calling {function_name} with {args}...")
            
            # Execute the actual logic
            if function_name == "generate_portfolio":
                results = generate_portfolio(df, n=args.get("n", 5))
            elif function_name == "budget_search":
                results = budget_search(df, limit=args.get("limit"))
            
            # Send the data back to the AI for the final explanation
            final_response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": user_input},
                    message,
                    {
                        "role": "tool",
                        "content": results.to_json(),
                        "tool_call_id": tool_call.id
                    }
                ]
            )
            return final_response.choices[0].message.content
            
    return message.content

print("✅ ChatOnSSET Intelligence Loop is Active.")

✅ ChatOnSSET Intelligence Loop is Active.


In [11]:
# Natural Language
query = "I have a $2.5 million grant. Where should I invest it in the Sofia region for the highest impact?"
print(chat_onsset(query))

🛠️ AI DECISION: Calling budget_search with {'limit': 2500000}...
With your $2.5 million grant, you can make impactful investments in the Sofia region by considering the following settlements:

1. **Settlement ID: 17144**
   - **Investment Required:** $2,264,269
   - **Population by 2025:** Approximately 6,950
   - **AgroLink Score:** 72,978.94
   
2. **Settlement ID: 17300**
   - **Investment Required:** $1,727,860
   - **Population by 2025:** Approximately 5,303
   - **AgroLink Score:** 65,294.45

3. **Settlement ID: 17073**
   - **Investment Required:** $2,309,673
   - **Population by 2025:** Approximately 7,089
   - **AgroLink Score:** 62,920.45

These locations were selected based on a balance of predicted population growth by 2030, high solar potential (Global Horizontal Irradiance - GHI), and the presence of social infrastructure, ensuring your investment achieves the highest possible impact.


In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('madagascar_sofia_calibrated.csv')

# Sort and print every single column line-by-line
for col in sorted(df.columns):
    print(col)


Index(['AgriDemand', 'CommercialDemand', 'Country', 'CurrentHVLineDist',
       'CurrentMVLineDist', 'EducationDemand', 'GHI', 'GridCellArea',
       'HealthDemand', 'HydropowerDist',
       ...
       'ElectrificationOrder', 'PerHouseholdDemand', 'ElecPop', 'Admin_1',
       'Hydropower', 'HydropowerFID', 'MGDist', 'Social_Anchor',
       'AgroLink_Potential', 'AgroLink_Score'],
      dtype='str', length=167)


In [2]:
# This will find any column that has 'lat', 'lon', 'x', or 'y' in the name
coord_cols = [c for c in df.columns if any(ext in c.lower() for ext in ['lat', 'lon', 'x_deg', 'y_deg'])]
print(coord_cols)

['X_deg', 'Y_deg']
